# MW-024 · Cyclohexan im MOL_LAB (Vertiefung)

**Die Cyclohexan-Struktur ist voreingestellt.** Klicke auf ▶ der Formularzelle. Das Moleküllabor lädt unterhalb der Zelle, ohne dass du Programmcode bearbeiten musst.

Die allgemeine Konformersuche liefert **Energie-Minima**, nicht automatisch die instabile Wanne oder den Halbsessel. Den gezielten Vierformen-Vergleich findest du in [DIDACTIC V0.8](https://colab.research.google.com/github/ekerzendorfer/MOL_LAB_DIDACTIC/blob/main/MOL_LAB_DIDACTIC_v0_4_2.ipynb).

*Freigegebene MOL_LAB-Version auf main (25.11.2026).*


In [ ]:
# @title ▶ MW-024 · Cyclohexan untersuchen { display-mode: "form" }
# @markdown **1)** Cyclohexan ist bereits vorausgewählt; andere SMILES können eingegeben werden
# @markdown **2)** Diese Zelle mit ▶ ausführen
# @markdown **3)** Das Moleküllabor erscheint unterhalb der Zelle

# @markdown ---
START_SMILES = "C1CCCCC1"  # @param {type:"string"}
START_NAME = "Cyclohexan"  # @param {type:"string"}
AUTO_LOAD_2D = True  # @param {type:"boolean"}
# @markdown ---

import sys, subprocess, os, stat, shutil, urllib.request, urllib.error

# ---- 1) Install (robust) ----
def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["py3Dmol", "ipywidgets"])

try:
    import rdkit
    print("✅ RDKit vorhanden:", rdkit.__version__)
except Exception:
    try:
        pip_install(["rdkit"])
        import rdkit
        print("✅ RDKit via pip:", rdkit.__version__)
    except Exception:
        print("⚠️ RDKit via pip nicht verfügbar → micromamba fallback ...")
        os.makedirs("/content/bin", exist_ok=True)
        subprocess.check_call("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content", shell=True)

        candidates = ["/content/bin/micromamba", "/content/bin/bin/micromamba", "/content/micromamba"]
        src = next((c for c in candidates if os.path.isfile(c)), None)
        if not src:
            raise RuntimeError("micromamba konnte nicht installiert werden (curl/tar).")

        dst = "/content/bin/micromamba"
        if src != dst:
            shutil.move(src, dst)
        os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

        prefix = "/content/rdkit_prefix"
        subprocess.check_call([dst, "create", "-y", "-p", prefix, "-c", "conda-forge", "python=3.10", "rdkit"],
                              stdout=subprocess.DEVNULL)

        site = f"{prefix}/lib/python3.10/site-packages"
        if site not in sys.path:
            sys.path.insert(0, site)

        import rdkit
        print("✅ RDKit via conda-forge:", rdkit.__version__)

# ---- 2) App-Code aus GitHub laden ----
USER = "ekerzendorfer"
REPO = "MOL_LAB"
PYFILE = "molekuellabor.py"

def try_download(branch):
    url = f"https://raw.githubusercontent.com/{USER}/{REPO}/{branch}/{PYFILE}"
    try:
        urllib.request.urlretrieve(url, PYFILE)
        print(f"✅ Geladen aus GitHub: {url}")
        return True
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return False
        raise

ok = try_download("main")
if not ok:
    raise FileNotFoundError(f"{PYFILE} auf main nicht gefunden in {USER}/{REPO}.")

# ---- 2.5) URL-Parameter auslesen (falls Start aus Aufgaben-Tool) ----
# Erwartet z.B.: ...ipynb?smiles=CCO&name=Ethanol&autoload2d=1
try:
    from google.colab import output
    import urllib.parse

    href = output.eval_js("window.location.href")
    parsed = urllib.parse.urlparse(href)
    qs = urllib.parse.parse_qs(parsed.query)

    url_smiles = qs.get("smiles", [""])[0].strip()
    url_name   = qs.get("name", [""])[0].strip()
    url_auto   = qs.get("autoload2d", [""])[0].strip().lower()

    # Falls Parameter gesetzt sind, überschreiben sie die @param-Werte
    if url_smiles:
        START_SMILES = url_smiles
    if url_name:
        START_NAME = url_name
    if url_auto in ("1", "true", "yes", "on"):
        AUTO_LOAD_2D = True
    elif url_auto in ("0", "false", "no", "off"):
        AUTO_LOAD_2D = False

    print("🔗 URL-Params:", "smiles" if url_smiles else "-", "name" if url_name else "-", f"autoload2d={AUTO_LOAD_2D}")
except Exception as e:
    # Wenn nicht in Colab oder JS nicht verfügbar → einfach ignorieren
    print("ℹ️ URL-Params nicht gelesen (ok):", type(e).__name__)

# ---- 3) Start-Parameter für die App verfügbar machen ----
# (Diese Variablen werden von molekuellabor.py gelesen, falls vorhanden.)
globals()["MOLLAB_START_SMILES"] = START_SMILES
globals()["MOLLAB_START_NAME"] = START_NAME
globals()["MOLLAB_AUTO_LOAD_2D"] = AUTO_LOAD_2D

# ---- 4) Start ----
exec(open(PYFILE, "r", encoding="utf-8").read())

# ---- 5) Startwerte sicher anwenden (robust, unabhängig vom App-Code) ----
# Erwartung: molekuellabor.py erzeugt ein globales Objekt namens "app"
try:
    app = globals().get("app", None)
    if app is None:
        raise RuntimeError("Kein globales 'app' Objekt gefunden. Prüfe, ob molekuellabor.py app = MoleculeApp() erzeugt.")

    # Das Beispiel-Dropdown mit dem vorausgewählten Startmolekül
    # synchronisieren (besonders MW-024), ohne freie SMILES zu verändern.
    if hasattr(app, "dd_example") and hasattr(app, "examples"):
        matched_example = next(
            (key for key, item in app.examples.items()
             if item.get("smiles") == START_SMILES), None
        )
        if matched_example:
            app.dd_example.value = matched_example

    # Felder setzen (nur wenn vorhanden)
    if hasattr(app, "input_smiles"):
        app.input_smiles.value = START_SMILES
    if hasattr(app, "input_name"):
        app.input_name.value = START_NAME

    # Auto-Load 2D ausführen
    if AUTO_LOAD_2D:
        if hasattr(app, "load_2d"):
            app.load_2d(None)  # Button-Callback direkt
        elif hasattr(app, "btn_load"):
            app.btn_load.click()  # alternativ Button klicken
        else:
            print("⚠️ Konnte 2D-Autoload nicht triggern (weder load_2d noch btn_load gefunden).")

    print("✅ Startwerte angewendet:", START_NAME, "|", START_SMILES[:80], "...")
except Exception as e:
    print("⚠️ Startwerte konnten nicht automatisch angewendet werden:", type(e).__name__, e)

